In [ ]:
# Codigo 01 - ok ================================================
# Rodar 2x: para month=7 e para month=12
# ===================== CONFIG (apenas L3m) =====================
ROOT = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados"
# ROOT = "./Dados"
L3M_DIR = f"{ROOT}/DAY/L3m"     # só L3m
ANOS = (2002, 2025)             # intervalo inclusivo
MONTH = 12                      # << mês desejado: 1..12 (1=jan, 7=jul, 12=dez)
COMPRESS_GZ = True              # salva como .csv.gz; se quiser .csv normal: False
VARS_TO_KEEP = None             # ex.: ["sst"] para exportar só TSM (opcional)

# Recorte geográfico do Estreito de Cingapura (lon_min, lon_max, lat_min, lat_max)
BBOX = (103.315900, 104.557400, 0.904561, 1.549800)
# ===============================================================

# (0) Montar Drive (Colab)
try:
    from google.colab import drive
    print("↪ Montando Google Drive...")
    drive.mount('/content/drive')
except Exception:
    print("↪ Ambiente genérico (pular montagem de Drive).")

import os, glob, re, traceback
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime

print("\n▶ Passo 1/6: Preparando utilitários...")

def ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path

def parse_date_from_name(fname: str):
    """Extrai AAAAMMDD do nome (formato Aqua-MODIS)."""
    m = re.search(r"\.(\d{8})\.", os.path.basename(fname))
    if not m: return None
    s = m.group(1)
    try:
        return datetime(int(s[:4]), int(s[4:6]), int(s[6:8]))
    except Exception:
        return None

def _parse_any_datetime(s: str):
    if not s: return None
    s = str(s).strip().replace("Z","")
    try:
        return pd.to_datetime(s, utc=False, errors="raise").to_pydatetime()
    except Exception:
        return None

def extract_datetime(ds: xr.Dataset, fname: str):
    """Prioriza attrs → variável time → nome do arquivo."""
    for k in ["time_coverage_start", "time_start", "start_time", "date_created"]:
        dt = _parse_any_datetime(ds.attrs.get(k))
        if dt: return dt
    if "time" in ds.variables:
        try:
            tvar = ds["time"]
            if tvar.ndim == 0:
                dt = pd.to_datetime(pd.Index([tvar.values]), errors="coerce")[0]
            else:
                dt = pd.to_datetime(pd.Index([tvar.values[0]]), errors="coerce")[0]
            if pd.notnull(dt): return pd.Timestamp(dt).to_pydatetime()
        except Exception:
            pass
    return parse_date_from_name(fname)

def open_ds_any(f):
    try:
        return xr.open_dataset(
            f, engine="h5netcdf", decode_cf=True, mask_and_scale=True,
            backend_kwargs={"phony_dims":"sort"}
        )
    except Exception:
        try:
            return xr.open_dataset(f, engine="scipy", decode_cf=True, mask_and_scale=True)
        except Exception:
            return xr.open_dataset(f, decode_cf=True, mask_and_scale=True)

def dataset_to_df_generic(ds, prefer_vars=None):
    """xr.Dataset -> DataFrame (mantém variáveis >0D; senão, linha escalar)."""
    ds = ds.reset_coords(drop=True)
    if prefer_vars:
        keep = [v for v in prefer_vars if v in ds.data_vars]
        if keep: ds = ds[keep]
    non_scalar = [v for v in ds.data_vars if ds[v].ndim > 0]
    if non_scalar:
        return ds[non_scalar].to_dataframe().reset_index()
    row = {}
    for v in ds.data_vars:
        val = ds[v].values
        try:
            row[v] = val.item() if getattr(val, "shape", ()) == () else np.asarray(val).ravel()[0].item()
        except Exception:
            row[v] = str(val)
    return pd.DataFrame([row]) if row else pd.DataFrame()

def collect_month_nc_l3m(in_dir: str, anos=(2002,2024), month=7):
    """Lista .nc do mês especificado no período, apenas em L3m, retornando DF com meta."""
    ncs = sorted(glob.glob(os.path.join(in_dir, "*.nc")))
    rows = []
    for f in ncs:
        dt = parse_date_from_name(f)
        ok = (dt is not None and anos[0] <= dt.year <= anos[1] and dt.month == month)
        rows.append({
            "folder": in_dir,
            "basename": os.path.basename(f),
            "fullpath": f,
            "dt": dt,
            "is_month_in_range": ok
        })
    df = pd.DataFrame(rows)
    return df[df["is_month_in_range"] == True].copy() if not df.empty else pd.DataFrame()

def check_missing_for_folder_l3m(in_dir: str):
    """Retorna DF de pendências do mês (sem csv/csv.gz) em L3m."""
    m_df = collect_month_nc_l3m(in_dir, anos=ANOS, month=MONTH)
    if m_df.empty:
        return pd.DataFrame(), pd.DataFrame()
    out_dir = os.path.join(in_dir, "csv_export")
    os.makedirs(out_dir, exist_ok=True)
    miss_rows, ok_rows = [], []
    for _, row in m_df.iterrows():
        base = os.path.splitext(row["basename"])[0]
        gz = os.path.join(out_dir, base + ".csv.gz")
        plain = os.path.join(out_dir, base + ".csv")
        if os.path.isfile(gz) or os.path.isfile(plain):
            ok_rows.append(row.to_dict())
        else:
            miss_rows.append(row.to_dict())
    return pd.DataFrame(miss_rows), pd.DataFrame(ok_rows)

def apply_bbox_df(df: pd.DataFrame, bbox):
    """Aplica BBOX quando colunas de lat/lon existirem."""
    if bbox is None:
        return df, False
    lon_min, lon_max, lat_min, lat_max = bbox
    lon_cols = [c for c in ["lon","longitude","x"] if c in df.columns]
    lat_cols = [c for c in ["lat","latitude","y"] if c in df.columns]
    if not lon_cols or not lat_cols:
        return df, False
    lon_c, lat_c = lon_cols[0], lat_cols[0]
    mask = (df[lon_c] >= lon_min) & (df[lon_c] <= lon_max) & \
           (df[lat_c] >= lat_min) & (df[lat_c] <= lat_max)
    return df.loc[mask].copy(), True

print("   OK utilitários prontos.")

# ------------------------- DETECÇÃO DE PENDÊNCIAS -------------------------
print(f"\n▶ Passo 2/6: Checando pendências do mês {MONTH} (2002–2024) em L3m...")
if not os.path.isdir(L3M_DIR):
    raise SystemExit(f"⛔ Pasta L3m não encontrada: {L3M_DIR}")

miss_df, ok_df = check_missing_for_folder_l3m(L3M_DIR)
print(f"   - Arquivos no mês no período: {len(miss_df) + len(ok_df)}")
print(f"   - Já convertidos: {len(ok_df)} | Pendentes: {len(miss_df)}")

# ------------------------- CONVERTER PENDÊNCIAS (L3m) -------------------------
print(f"\n▶ Passo 3/6: Convertendo SOMENTE pendências do mês {MONTH} em L3m (com BBOX)...")
conv_ok = 0
conv_fail = 0
converted_paths = []
bbox_used_count = 0

if miss_df.empty:
    print("   ✓ Não há pendências em L3m. Nada a converter.")
else:
    for _, row in miss_df.iterrows():
        f = row["fullpath"]
        base = os.path.splitext(os.path.basename(f))[0]
        out_dir = ensure_dir(os.path.join(L3M_DIR, "csv_export"))
        out_ext = ".csv.gz" if COMPRESS_GZ else ".csv"
        out_path = os.path.join(out_dir, base + out_ext)
        print(f"   → Convertendo: {os.path.basename(f)} (L3m)")

        try:
            ds = open_ds_any(f)
            dt = extract_datetime(ds, f)
            # Segurança: reconfirmar mês/período
            if dt is None or not (ANOS[0] <= dt.year <= ANOS[1]) or dt.month != MONTH:
                print("     [skip] Data não pertence ao mês/período configurado.")
                ds.close()
                continue

            # Converte para DataFrame (grade regular)
            df = dataset_to_df_generic(ds, prefer_vars=VARS_TO_KEEP)

            # Reorganiza: coords primeiro
            lead_coords = [c for c in ["lat","latitude","y","i","j"] if c in df.columns] + \
                          [c for c in ["lon","longitude","x","row","col","bin"] if c in df.columns]
            data_cols = [c for c in df.columns if c not in lead_coords]
            if data_cols:
                df = df[lead_coords + data_cols]

            # Aplica BBOX (se possível)
            df, used = apply_bbox_df(df, BBOX)
            bbox_used_count += int(used)
            if BBOX is not None and not used:
                print("     [aviso] BBOX definido, mas não aplicado (colunas lat/lon não detectadas).")

            # carimbo temporal detalhado
            Y, M, D = dt.year, dt.month, dt.day
            h, mi, s = dt.hour, dt.minute, dt.second
            df.insert(0, "year",   Y)
            df.insert(1, "month",  M)
            df.insert(2, "day",    D)
            df.insert(3, "hour",   h)
            df.insert(4, "minute", mi)
            df.insert(5, "second", s)
            df.insert(6, "date", f"{Y:04d}-{M:02d}-{D:02d}")
            df.insert(7, "datetime_iso", f"{Y:04d}-{M:02d}-{D:02d}T{h:02d}:{mi:02d}:{s:02d}")

            # ordenar por tempo (quando fizer sentido)
            try:
                df = df.sort_values(["year","month","day","hour","minute","second"])
            except Exception:
                pass

            # salvar
            if COMPRESS_GZ:
                df.to_csv(out_path, index=False, compression="infer")
            else:
                df.to_csv(out_path, index=False)

            converted_paths.append(out_path)
            conv_ok += 1
            print(f"     ✓ salvo: {out_path} (linhas: {len(df)})")
            ds.close()

        except Exception as e:
            conv_fail += 1
            print(f"     [ERRO] {os.path.basename(f)} -> {e}")
            traceback.print_exc(limit=1)

# ------------------------- REVALIDAÇÃO -------------------------
print(f"\n▶ Passo 4/6: Revalidando conversões do mês {MONTH} em L3m...")
miss2_df, ok2_df = check_missing_for_folder_l3m(L3M_DIR)
print(f"   Pendências restantes: {len(miss2_df)}")

# ------------------------- RESUMO -------------------------
print("\n▶ Passo 5/6: Resumo (L3m)")
print(f"  Convertidos agora          : {conv_ok}")
print(f"  Falhas na conversão        : {conv_fail}")
print(f"  Pendências finais (L3m)    : {len(miss2_df)}")
if BBOX is not None:
    print(f"  BBOX aplicado (arquivos)   : {bbox_used_count}")

# ------------------------- LISTAR NOVOS/ PENDÊNCIAS -------------------------
print("\n▶ Passo 6/6: Listando novos arquivos gerados e pendências finais (se houver)...")
if converted_paths:
    print("  Arquivos gerados (amostra):")
    for p in converted_paths[:20]:
        print("   -", p)
else:
    print("  (Nenhum novo arquivo criado nesta execução.)")

# opcional: salvar CSV com pendências restantes
if len(miss2_df) > 0:
    out_missing = os.path.join(L3M_DIR, "csv_export", f"pendencias_mes_{MONTH:02d}_sem_csv_restantes_L3m.csv")
    try:
        os.makedirs(os.path.dirname(out_missing), exist_ok=True)
        miss2_df.to_csv(out_missing, index=False)
        print("  ✓ Pendências restantes salvas em:", out_missing)
    except Exception as e:
        print("  ⚠ Falha ao salvar pendências restantes em", out_missing, "->", e)

print("\n✅ FIM (L3m, mês =", MONTH, ")")


In [ ]:
# Codigo 02 - ok ================================================
# ==============================================================================
# PIPELINE FLEXÍVEL: CONVERSÃO AQUA_MODIS -> .csv.gz (Estreito de Cingapura)
# ==============================================================================

# ===================== CONFIGURAÇÃO DO PIPELINE =====================
# ROOT = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados"
ROOT = "./Dados"
INPUT_DIR = f"{ROOT}/MO/L3m"        # Diretório contendo os arquivos .nc
OUTPUT_DIR = f"{INPUT_DIR}/csv_export"
ANOS = (2002, 2025)                 # Intervalo inclusivo de anos desejado

# Se quiser filtrar por um mês específico, basta descomentar a linha indicada
# no "Passo 2/5" mais abaixo no código.
MONTH = 12                          # Mês alvo (usado apenas se o filtro for ativado)

# Recorte geográfico estrito para o Estreito de Cingapura
BBOX = (103.315900, 104.557400, 0.904561, 1.549800)
# ====================================================================

# (0) Conectar ao ambiente do Google Drive (Colab)
try:
    from google.colab import drive
    print("↪ Montando Google Drive...")
    drive.mount('/content/drive')
except Exception:
    print("↪ Ambiente local ou genérico detectado (pulando montagem).")

import os, glob, re, traceback
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime

print("\n▶ Passo 1/5: Inicializando funções utilitárias...")

def parse_date_from_name(fname: str):
    """
    Extrai a data de arquivos padrão AQUA_MODIS.
    Suporta o formato de intervalo composto com underline (Ex: .20021201_20021231.).
    """
    basename = os.path.basename(fname)

    # 1. Captura os 8 primeiros dígitos de um intervalo separado por underline
    m_range = re.search(r"\.(\d{8})_\d{8}\.", basename)
    if m_range:
        s = m_range.group(1)
        try: return datetime(int(s[:4]), int(s[4:6]), int(s[6:8]))
        except Exception: pass

    # 2. Busca padrão clássico de 8 dígitos isolados
    m_eight = re.search(r"\.(\d{8})\.", basename)
    if m_eight:
        s = m_eight.group(1)
        try: return datetime(int(s[:4]), int(s[4:6]), int(s[6:8]))
        except Exception: pass

    # 3. Busca padrão Juliano/Ordinal (Ex: A2023335)
    m_julian = re.search(r"[AT](\d{7})\.", basename)
    if m_julian:
        s = m_julian.group(1)
        try:
            year = int(s[:4])
            day_of_year = int(s[4:])
            return datetime.strptime(f"{year}-{day_of_year}", "%Y-%j")
        except Exception: pass

    return None

def apply_bbox_xarray(ds: xr.Dataset, bbox):
    """Filtra as coordenadas espaciais direto no xarray antes de gerar o DataFrame (Otimiza RAM)."""
    if bbox is None:
        return ds
    lon_min, lon_max, lat_min, lat_max = bbox

    lon_name = [dim for dim in ["lon", "longitude", "x"] if dim in ds.dims or dim in ds.coords]
    lat_name = [dim for dim in ["lat", "latitude", "y"] if dim in ds.dims or dim in ds.coords]

    if not lon_name or not lat_name:
        return ds
    lon_c, lat_c = lon_name[0], lat_name[0]

    try:
        lat_values = ds[lat_c].values
        if len(lat_values) > 1 and lat_values[0] > lat_values[1]:  # Matriz Norte -> Sul (Invertida)
            ds = ds.sel({lon_c: slice(lon_min, lon_max), lat_c: slice(lat_max, lat_min)})
        else:  # Matriz Sul -> Norte (Crescente)
            ds = ds.sel({lon_c: slice(lon_min, lon_max), lat_c: slice(lat_min, lat_max)})
    except Exception:
        pass
    return ds

# --- PROCESSAMENTO PRINCIPAL ---
print(f"\n▶ Passo 2/5: Mapeando arquivos e checando pendências...")
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_nc_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.nc")))
pending_files = []

for f in all_nc_files:
    dt = parse_date_from_name(f)

    # Valida o intervalo de anos
    if dt and (ANOS[0] <= dt.year <= ANOS[1]):

        # ==================================================================
        # OPÇÃO DE FILTRO POR MÊS:
        # Por padrão (com a linha abaixo comentada), ele processa TODOS os meses.
        # Se quiser filtrar apenas pelo mês configurado na variável MONTH lá no topo,
        # basta remover o símbolo '#' do início da linha abaixo:
        # if dt.month != MONTH: continue
        # ==================================================================

        base_name = os.path.splitext(os.path.basename(f))[0]
        out_path = os.path.join(OUTPUT_DIR, f"{base_name}.csv.gz")

        # Sistema de Checkpoint: não reprocessa arquivos que já existem na pasta de destino
        if not os.path.isfile(out_path):
            pending_files.append((f, out_path, dt))

print(f"   - Total de arquivos .nc detectados na pasta: {len(all_nc_files)}")
print(f"   - Total de arquivos na fila para conversão: {len(pending_files)}")

print(f"\n▶ Passo 3/5: Iniciando o loop de conversão de dados...")
conv_sucesso = 0

for in_f, out_f, dt_obj in pending_files:
    print(f" ↪ Processando: {os.path.basename(in_f)}")
    try:
        ds = xr.open_dataset(in_f, decode_cf=True, mask_and_scale=True)
        ds = apply_bbox_xarray(ds, BBOX)

        df = ds.to_dataframe().reset_index()

        if df.empty:
            print("   ⚠️ [Aviso] O recorte geográfico resultou em uma tabela vazia para este arquivo.")
            ds.close()
            continue

        if hasattr(df, 'reset_coords'):
            df = df.reset_coords(drop=True)

        # Injeção temporal dinâmica obtida de cada arquivo individual
        Y, M, D = dt_obj.year, dt_obj.month, dt_obj.day
        df.insert(0, "year", Y)
        df.insert(1, "month", M)
        df.insert(2, "day", D)
        df.insert(3, "date", f"{Y:04d}-{M:02d}-{D:02d}")

        # Reorganiza a estrutura das colunas (Tempo -> Espaço -> Variáveis)
        coords = [c for c in ["lat", "latitude", "lon", "longitude"] if c in df.columns]
        datas = [c for c in df.columns if c not in coords and c not in ["year", "month", "day", "date"]]
        df = df[["year", "month", "day", "date"] + coords + datas]

        if not out_f.endswith(".csv.gz"):
            base_limpa = out_f.split(".nc")[0].split(".csv")[0]
            out_f = f"{base_limpa}.csv.gz"

        df.to_csv(out_f, index=False, compression="gzip")
        ds.close()
        conv_sucesso += 1
        print(f"   ✓ Arquivo salvo com sucesso: {os.path.basename(out_f)}")

    except Exception as e:
        print(f"   ❌ Falha crítica ao converter {os.path.basename(in_f)}: {e}")
        traceback.print_exc(limit=1)

# --- REVALIDAÇÃO E RESUMO ---
print("\n▶ Passo 4/5: Validando integridade do lote final...")
arquivos_finais = glob.glob(os.path.join(OUTPUT_DIR, "*.csv.gz"))
print(f"   - Arquivos compactados salvos atualmente em '{os.path.basename(OUTPUT_DIR)}': {len(arquivos_finais)}")

print("\n▶ Passo 5/5: Resumo da Execução")
print(f"   - Convertidos com sucesso nesta rodada : {conv_sucesso}")
print(f"   - Total de pendências restantes        : {len(pending_files) - conv_sucesso}")
print(f"\n✅ PIPELINE CONCLUÍDO")

In [ ]:
# Codigo 03 - ok ================================================
# ==============================================================================
# PIPELINE FLEXÍVEL: CONVERSÃO AQUA_MODIS ANUAL (YR) -> .csv.gz
# ==============================================================================

# ===================== CONFIGURAÇÃO DO PIPELINE =====================
ROOT = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados"
INPUT_DIR = f"{ROOT}/YR/L3m"        # Diretório contendo os arquivos .nc anuais
OUTPUT_DIR = f"{INPUT_DIR}/csv_export"
ANOS = (2002, 2025)                 # Intervalo inclusivo de anos desejado

# Se você quiser processar apenas um ano específico no futuro (ex: apenas 2002),
# você pode definir a variável abaixo e ativar o filtro no "Passo 2/5".
ANO_ALVO = 2002
# ====================================================================

# Recorte geográfico estrito para o Estreito de Cingapura
BBOX = (103.315900, 104.557400, 0.904561, 1.549800)

# (0) Conectar ao ambiente do Google Drive (Colab)
try:
    from google.colab import drive
    print("↪ Montando Google Drive...")
    drive.mount('/content/drive')
except Exception:
    print("↪ Ambiente local ou genérico detectado (pulando montagem).")

import os, glob, re, traceback
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime

print("\n▶ Passo 1/5: Inicializando funções utilitárias...")

def parse_date_from_name(fname: str):
    """
    Extrai a data de arquivos padrão AQUA_MODIS anuais.
    Suporta o formato de intervalo composto com underline (Ex: .20020101_20021231.).
    """
    basename = os.path.basename(fname)

    # 1. Captura os 8 primeiros dígitos de um intervalo separado por underline
    m_range = re.search(r"\.(\d{8})_\d{8}\.", basename)
    if m_range:
        s = m_range.group(1)
        try: return datetime(int(s[:4]), int(s[4:6]), int(s[6:8]))
        except Exception: pass

    # 2. Busca padrão clássico de 8 dígitos isolados
    m_eight = re.search(r"\.(\d{8})\.", basename)
    if m_eight:
        s = m_eight.group(1)
        try: return datetime(int(s[:4]), int(s[4:6]), int(s[6:8]))
        except Exception: pass

    return None

def apply_bbox_xarray(ds: xr.Dataset, bbox):
    """Filtra as coordenadas espaciais direto no xarray antes de gerar o DataFrame (Otimiza RAM)."""
    if bbox is None:
        return ds
    lon_min, lon_max, lat_min, lat_max = bbox

    lon_name = [dim for dim in ["lon", "longitude", "x"] if dim in ds.dims or dim in ds.coords]
    lat_name = [dim for dim in ["lat", "latitude", "y"] if dim in ds.dims or dim in ds.coords]

    if not lon_name or not lat_name:
        return ds
    lon_c, lat_c = lon_name[0], lat_name[0]

    try:
        lat_values = ds[lat_c].values
        if len(lat_values) > 1 and lat_values[0] > lat_values[1]:  # Matriz Norte -> Sul (Invertida)
            ds = ds.sel({lon_c: slice(lon_min, lon_max), lat_c: slice(lat_max, lat_min)})
        else:  # Matriz Sul -> Norte (Crescente)
            ds = ds.sel({lon_c: slice(lon_min, lon_max), lat_c: slice(lat_min, lat_max)})
    except Exception:
        pass
    return ds

# --- PROCESSAMENTO PRINCIPAL ---
print(f"\n▶ Passo 2/5: Mapeando arquivos e checando pendências para o intervalo {ANOS[0]}-{ANOS[1]}...")
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_nc_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.nc")))
pending_files = []

for f in all_nc_files:
    dt = parse_date_from_name(f)

    # Valida se o arquivo está dentro do intervalo geral de anos configurado
    if dt and (ANOS[0] <= dt.year <= ANOS[1]):

        # ==================================================================
        # OPÇÃO DE FILTRO POR ANO INDIVIDUAL:
        # Por padrão (com a linha abaixo comentada), converte TODOS os anos do intervalo.
        # Se quiser filtrar APENAS pelo ano configurado em ANO_ALVO, remova o '#' abaixo:
        # if dt.year != ANO_ALVO: continue
        # ==================================================================

        base_name = os.path.splitext(os.path.basename(f))[0]
        out_path = os.path.join(OUTPUT_DIR, f"{base_name}.csv.gz")

        # Sistema de Checkpoint: pula se o .csv.gz processado já existir
        if not os.path.isfile(out_path):
            pending_files.append((f, out_path, dt))

print(f"   - Total de arquivos .nc detectados na pasta YR: {len(all_nc_files)}")
print(f"   - Total de arquivos anuais pendentes na fila: {len(pending_files)}")

print(f"\n▶ Passo 3/5: Iniciando o loop de conversão de dados...")
conv_sucesso = 0

for in_f, out_f, dt_obj in pending_files:
    print(f" ↪ Processando: {os.path.basename(in_f)}")
    try:
        ds = xr.open_dataset(in_f, decode_cf=True, mask_and_scale=True)
        ds = apply_bbox_xarray(ds, BBOX)

        df = ds.to_dataframe().reset_index()

        if df.empty:
            print("   ⚠️ [Aviso] O recorte geográfico resultou em uma tabela vazia para este arquivo.")
            ds.close()
            continue

        if hasattr(df, 'reset_coords'):
            df = df.reset_coords(drop=True)

        # Injeção temporal com base no ano de início extraído do nome do arquivo
        Y = dt_obj.year
        df.insert(0, "year", Y)
        df.insert(1, "date", f"{Y:04d}-01-01") # Define data padrão de início do bloco anual

        # Reorganiza a estrutura das colunas (Tempo -> Espaço -> Variáveis de dados)
        coords = [c for c in ["lat", "latitude", "lon", "longitude"] if c in df.columns]
        datas = [c for c in df.columns if c not in coords and c not in ["year", "date"]]
        df = df[["year", "date"] + coords + datas]

        if not out_f.endswith(".csv.gz"):
            base_limpa = out_f.split(".nc")[0].split(".csv")[0]
            out_f = f"{base_limpa}.csv.gz"

        df.to_csv(out_f, index=False, compression="gzip")
        ds.close()
        conv_sucesso += 1
        print(f"   ✓ Arquivo salvo com sucesso: {os.path.basename(out_f)}")

    except Exception as e:
        print(f"   ❌ Falha crítica ao converter {os.path.basename(in_f)}: {e}")
        traceback.print_exc(limit=1)

# --- REVALIDAÇÃO E RESUMO ---
print("\n▶ Passo 4/5: Validando integridade do lote final...")
arquivos_finais = glob.glob(os.path.join(OUTPUT_DIR, "*.csv.gz"))
print(f"   - Arquivos compactados salvos atualmente em '{os.path.basename(OUTPUT_DIR)}': {len(arquivos_finais)}")

print("\n▶ Passo 5/5: Resumo da Execução")
print(f"   - Convertidos com sucesso nesta rodada : {conv_sucesso}")
print(f"   - Total de pendências restantes        : {len(pending_files) - conv_sucesso}")
print(f"\n✅ PIPELINE ANUAL CONCLUÍDO")

In [ ]:
# Codigo 04 - ok ================================================
# conversao escolhendo o mes ===================================================
# ==============================================================================
# PIPELINE COMPLETO CORRIGIDO: CONVERSÃO AQUA_MODIS -> .csv.gz (Estreito de Cingapura)
# ==============================================================================

# ===================== CONFIGURAÇÃO DO PIPELINE =====================
ROOT = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados"
INPUT_DIR = f"{ROOT}/MO/L3m"        # Diretório contendo os arquivos .nc
OUTPUT_DIR = f"{INPUT_DIR}/csv_export"
ANOS = (2002, 2025)                 # Intervalo inclusivo de anos desejado
MONTH = 7                           # Mês alvo do processamento (Ex: 12 = Dezembro)

# Recorte geográfico estrito para o Estreito de Cingapura
BBOX = (103.315900, 104.557400, 0.904561, 1.549800)
# ====================================================================

# (0) Conectar ao ambiente do Google Drive (Colab)
try:
    from google.colab import drive
    print("↪ Montando Google Drive...")
    drive.mount('/content/drive')
except Exception:
    print("↪ Ambiente local ou genérico detectado (pulando montagem).")

import os, glob, re, traceback
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime

print("\n▶ Passo 1/5: Inicializando funções utilitárias...")

def parse_date_from_name(fname: str):
    """
    Extrai a data de arquivos padrão AQUA_MODIS.
    Suporta os formatos de intervalo (AAAAMMDD_AAAAMMDD), 8 dígitos simples e Dia Juliano.
    """
    basename = os.path.basename(fname)

    # 1. Captura os 8 primeiros dígitos de um intervalo separado por underline (Ex: .20021201_20021231.)
    m_range = re.search(r"\.(\d{8})_\d{8}\.", basename)
    if m_range:
        s = m_range.group(1)
        try: return datetime(int(s[:4]), int(s[4:6]), int(s[6:8]))
        except Exception: pass

    # 2. Busca padrão clássico de 8 dígitos isolados
    m_eight = re.search(r"\.(\d{8})\.", basename)
    if m_eight:
        s = m_eight.group(1)
        try: return datetime(int(s[:4]), int(s[4:6]), int(s[6:8]))
        except Exception: pass

    # 3. Busca padrão Juliano/Ordinal (Ex: A2023335)
    m_julian = re.search(r"[AT](\d{7})\.", basename)
    if m_julian:
        s = m_julian.group(1)
        try:
            year = int(s[:4])
            day_of_year = int(s[4:])
            return datetime.strptime(f"{year}-{day_of_year}", "%Y-%j")
        except Exception: pass

    return None

def apply_bbox_xarray(ds: xr.Dataset, bbox):
    """Filtra as coordenadas espaciais direto no xarray antes de gerar o DataFrame (Otimiza RAM)."""
    if bbox is None:
        return ds
    lon_min, lon_max, lat_min, lat_max = bbox

    lon_name = [dim for dim in ["lon", "longitude", "x"] if dim in ds.dims or dim in ds.coords]
    lat_name = [dim for dim in ["lat", "latitude", "y"] if dim in ds.dims or dim in ds.coords]

    if not lon_name or not lat_name:
        return ds
    lon_c, lat_c = lon_name[0], lat_name[0]

    try:
        lat_values = ds[lat_c].values
        if len(lat_values) > 1 and lat_values[0] > lat_values[1]:  # Matriz Norte -> Sul (Invertida)
            ds = ds.sel({lon_c: slice(lon_min, lon_max), lat_c: slice(lat_max, lat_min)})
        else:  # Matriz Sul -> Norte (Crescente)
            ds = ds.sel({lon_c: slice(lon_min, lon_max), lat_c: slice(lat_min, lat_max)})
    except Exception:
        pass
    return ds

# --- PROCESSAMENTO PRINCIPAL ---
print(f"\n▶ Passo 2/5: Mapeando arquivos e checando pendências para o mês {MONTH:02d} ({ANOS[0]}-{ANOS[1]})...")
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_nc_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.nc")))
pending_files = []

for f in all_nc_files:
    dt = parse_date_from_name(f)

    # Filtra apenas arquivos que pertencem à janela temporal e ao mês configurado
    if dt and (ANOS[0] <= dt.year <= ANOS[1]) and (dt.month == MONTH):
        base_name = os.path.splitext(os.path.basename(f))[0]
        out_path = os.path.join(OUTPUT_DIR, f"{base_name}.csv.gz")

        # Sistema de Checkpoint (Idempotência): Só processa se a saída comprimida não existir
        if not os.path.isfile(out_path):
            pending_files.append((f, out_path, dt))

print(f"   - Total de arquivos .nc detectados na pasta: {len(all_nc_files)}")
print(f"   - Arquivos elegíveis para o mês {MONTH:02d} pendentes de conversão: {len(pending_files)}")

print(f"\n▶ Passo 3/5: Iniciando o loop de conversão de dados...")
conv_sucesso = 0

for in_f, out_f, dt_obj in pending_files:
    print(f" ↪ Processando: {os.path.basename(in_f)}")
    try:
        # Abre o dataset limpando flags CF e aplica corte BBOX nativo
        ds = xr.open_dataset(in_f, decode_cf=True, mask_and_scale=True)
        ds = apply_bbox_xarray(ds, BBOX)

        # Converte apenas o recorte geográfico para DataFrame Pandas
        df = ds.to_dataframe().reset_index()

        if df.empty:
            print("   ⚠️ [Aviso] O recorte geográfico resultou em uma tabela vazia para este arquivo.")
            ds.close()
            continue

        # Garante a eliminação de metadados de coordenadas crus
        if hasattr(df, 'reset_coords'):
            df = df.reset_coords(drop=True)

        # Insere colunas explícitas estruturadas de tempo
        Y, M, D = dt_obj.year, dt_obj.month, dt_obj.day
        df.insert(0, "year", Y)
        df.insert(1, "month", M)
        df.insert(2, "day", D)
        df.insert(3, "date", f"{Y:04d}-{M:02d}-{D:02d}")

        # Reorganiza a estrutura das colunas para melhor legibilidade (Tempo -> Espaço -> Variáveis)
        coords = [c for c in ["lat", "latitude", "lon", "longitude"] if c in df.columns]
        datas = [c for c in df.columns if c not in coords and c not in ["year", "month", "day", "date"]]
        df = df[["year", "month", "day", "date"] + coords + datas]

        # Força de forma explícita o sufixo correto de compressão no nome do arquivo
        if not out_f.endswith(".csv.gz"):
            base_limpa = out_f.split(".nc")[0].split(".csv")[0]
            out_f = f"{base_limpa}.csv.gz"

        # Exporta aplicando compressão explícita GZIP estruturada pelo Pandas
        df.to_csv(out_f, index=False, compression="gzip")
        ds.close()
        conv_sucesso += 1
        print(f"   ✓ Arquivo salvo com sucesso: {os.path.basename(out_f)}")

    except Exception as e:
        print(f"   ❌ Falha crítica ao converter {os.path.basename(in_f)}: {e}")
        traceback.print_exc(limit=1)

# --- REVALIDAÇÃO E RESUMO ---
print("\n▶ Passo 4/5: Validando integridade do lote atual...")
arquivos_finais = glob.glob(os.path.join(OUTPUT_DIR, "*.csv.gz"))
print(f"   - Arquivos compactados salvos atualmente em '{os.path.basename(OUTPUT_DIR)}': {len(arquivos_finais)}")

print("\n▶ Passo 5/5: Resumo da Execução")
print(f"   - Convertidos com sucesso nesta rodada : {conv_sucesso}")
print(f"   - Total de pendências restantes        : {len(pending_files) - conv_sucesso}")
print(f"\n✅ PIPELINE CONCLUÍDO (Lote Mês = {MONTH:02d})")

In [ ]:
# Codigo 05 - ok =====================================================
# === SST Daily Mean (NO smoothing) | X-axis ticks every 2 years | L3m
# Saves figure as PNG and SVG under csv_export/fig

# --- Month selection ---
#   - MONTH = 1..12 for a specific month (1=Jan, 7=Jul, 12=Dec)
#   - MONTH = None to use ALL months
MONTH = 12   # << change here (e.g., 7 for July, 12 for December, None = all)

# --- CSV directory (L3m) ---
CSV_DIR = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados/DAY/L3m/csv_export"

# --- Mount Google Drive (Colab) ---
try:
    from google.colab import drive
    print("↪ Mounting Google Drive...")
    drive.mount('/content/drive')
except Exception:
    print("↪ Non-Colab environment (skipping mount).")

import os, glob
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, DateFormatter
from datetime import datetime

# --------- 1) Check directory ----------
print("\n▶ Step 1/5: Checking directory...")
assert os.path.isdir(CSV_DIR), f"Directory not found: {CSV_DIR}"

# --------- 2) List files ----------
print("\n▶ Step 2/5: Listing .csv.gz/.csv files...")
files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv.gz"))) + \
        sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
print("   Files found:", len(files))
assert len(files) > 0, "No CSV files found."

# --------- Loader ----------
def load_one_csv(path):
    """Return DataFrame with 'date' (normalized datetime) and 'sst' (float)."""
    head = pd.read_csv(path, nrows=0)
    cols = list(head.columns)

    # SST column
    sst_col = None
    for cand in ["sst","sst4","sea_surface_temperature"]:
        if cand in cols:
            sst_col = cand; break
    if sst_col is None:
        sst_like = [c for c in cols if "sst" in c.lower()]
        sst_col = sst_like[0] if sst_like else None
    if sst_col is None:
        return None

    # Date column
    if "date" in cols:
        df = pd.read_csv(path, usecols=["date", sst_col])
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    else:
        need = [c for c in ["year","month","day","datetime_iso"] if c in cols]
        if not need:
            return None
        dft = pd.read_csv(path, usecols=need)
        if {"year","month","day"}.issubset(dft.columns):
            df = pd.DataFrame({
                "date": pd.to_datetime(dict(year=dft["year"], month=dft["month"], day=dft["day"]),
                                       errors="coerce")
            })
        elif "datetime_iso" in dft.columns:
            df = pd.DataFrame({"date": pd.to_datetime(dft["datetime_iso"], errors="coerce")})
        else:
            return None
        df[sst_col] = pd.read_csv(path, usecols=[sst_col])[sst_col]

    # Cleaning
    df = df.dropna(subset=["date"])
    df[sst_col] = pd.to_numeric(df[sst_col], errors="coerce")
    df = df.dropna(subset=[sst_col]).rename(columns={sst_col: "sst"})
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    return df[["date","sst"]]

# --------- 3) Read & stack ----------
print("\n▶ Step 3/5: Reading and stacking data...")
dfs, bad = [], 0
for f in files:
    try:
        dfx = load_one_csv(f)
        if dfx is None or dfx.empty:
            bad += 1
            continue
        dfs.append(dfx)
    except Exception:
        bad += 1
print("   Valid files read:", len(dfs), "| problematic:", bad)
assert len(dfs) > 0, "Could not read any valid CSV file."

data = pd.concat(dfs, ignore_index=True)

# Daily spatial mean
daily = (data.groupby("date", as_index=False)["sst"]
         .mean()
         .sort_values("date"))

# Month filter (optional)
months_en = {1:"January",2:"February",3:"March",4:"April",5:"May",6:"June",
             7:"July",8:"August",9:"September",10:"October",11:"November",12:"December"}
if MONTH is not None:
    daily = daily[daily["date"].dt.month == MONTH].copy()
    title_suffix = f" ({months_en[MONTH]})"
    month_slug = f"{MONTH:02d}"
else:
    title_suffix = " (all months)"
    month_slug = "allmonths"

print("   Time span:", daily["date"].min(), "→", daily["date"].max())
print("   # of daily points:", len(daily))
assert len(daily) > 0, "No data after the applied filter."

# --------- 4) Plot ----------
print("\n▶ Step 4/5: Plotting (NO smoothing, 2-year ticks)...")
plt.figure()
plt.plot(daily["date"], daily["sst"], linestyle="-")   # no moving average
plt.title("SST - Daily Mean" + title_suffix + " (X-axis ticks every 2 years) - L3m")
plt.xlabel("Year")
plt.ylabel("SST (°C)")
plt.grid(True)

ax = plt.gca()
ax.xaxis.set_major_locator(YearLocator(base=2))    # 2-year ticks
ax.xaxis.set_major_formatter(DateFormatter('%Y'))
plt.tight_layout()

# --------- 5) Save images ----------
print("\n▶ Step 5/5: Saving figures (PNG and SVG)...")
fig_dir = os.path.join(CSV_DIR, "fig")
os.makedirs(fig_dir, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
base_name = f"sst_daily_mean_L3m_{month_slug}_tick2years_{ts}"

png_path = os.path.join(fig_dir, base_name + ".png")
svg_path = os.path.join(fig_dir, base_name + ".svg")

plt.savefig(png_path, dpi=150)   # PNG
plt.savefig(svg_path)            # SVG
plt.show()

print("   ✓ PNG saved to:", png_path)
print("   ✓ SVG saved to:", svg_path)
print("\n✅ Done.")


In [ ]:
# Codigo 06 - ok =====================================================
# === SST Monthly Mean (NO smoothing) | X-axis ticks every 2 years | MO/L3m
# Saves figure as PNG and SVG under csv_export/fig

# --- Month selection across years ---
#   - MONTH = 1..12 for a specific month (1=Jan, 7=Jul, 12=Dec) → e.g., "July across 2002..2024"
#   - MONTH = None to use ALL months (full monthly series)
MONTH = 12   # << change here (e.g., 7 for July, 12 for December, None = all)

# --- CSV directory (MO/L3m) ---
CSV_DIR = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados/MO/L3m/csv_export"

# --- Mount Google Drive (Colab) ---
try:
    from google.colab import drive
    print("↪ Mounting Google Drive...")
    drive.mount('/content/drive')
except Exception:
    print("↪ Non-Colab environment (skipping mount).")

import os, glob
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, DateFormatter
from datetime import datetime

# --------- 1) Check directory ----------
print("\n▶ Step 1/5: Checking directory...")
assert os.path.isdir(CSV_DIR), f"Directory not found: {CSV_DIR}"

# --------- 2) List files ----------
print("\n▶ Step 2/5: Listing .csv.gz/.csv files...")
files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv.gz"))) + \
        sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
print("   Files found:", len(files))
assert len(files) > 0, "No CSV files found."

# --------- Loader (monthly) ----------
def load_one_csv_monthly(path):
    """
    Return DataFrame with:
      - 'month_date' = normalized monthly timestamp (YYYY-MM-01)
      - 'sst'        = SST values (float)
    Works whether the CSV has 'date' or 'year'/'month' or 'datetime_iso'.
    """
    head = pd.read_csv(path, nrows=0)
    cols = list(head.columns)

    # Detect SST column
    sst_col = None
    for cand in ["sst","sst4","sea_surface_temperature"]:
        if cand in cols:
            sst_col = cand; break
    if sst_col is None:
        sst_like = [c for c in cols if "sst" in c.lower()]
        sst_col = sst_like[0] if sst_like else None
    if sst_col is None:
        return None  # no SST column

    # Build a monthly 'month_date'
    if "date" in cols:
        df = pd.read_csv(path, usecols=["date", sst_col])
        dt = pd.to_datetime(df["date"], errors="coerce")
        # normalize to first day of month
        month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        out = pd.DataFrame({"month_date": month_date, "sst": pd.to_numeric(df[sst_col], errors="coerce")})
    else:
        # try year/month or datetime_iso
        need = [c for c in ["year","month","datetime_iso"] if c in cols]
        if not need:
            return None
        dft = pd.read_csv(path, usecols=need)
        if {"year","month"}.issubset(dft.columns):
            y = pd.to_numeric(dft["year"], errors="coerce")
            m = pd.to_numeric(dft["month"], errors="coerce")
            month_date = pd.to_datetime(dict(year=y, month=m, day=1), errors="coerce")
        elif "datetime_iso" in dft.columns:
            dt = pd.to_datetime(dft["datetime_iso"], errors="coerce")
            month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        else:
            return None
        sst_series = pd.read_csv(path, usecols=[sst_col])[sst_col]
        out = pd.DataFrame({"month_date": month_date, "sst": pd.to_numeric(sst_series, errors="coerce")})

    out = out.dropna(subset=["month_date","sst"])
    out["month_date"] = pd.to_datetime(out["month_date"]).dt.normalize()
    return out

# --------- 3) Read & stack ----------
print("\n▶ Step 3/5: Reading and stacking data...")
dfs, bad = [], 0
for f in files:
    try:
        dfx = load_one_csv_monthly(f)
        if dfx is None or dfx.empty:
            bad += 1
            continue
        dfs.append(dfx)
    except Exception:
        bad += 1
print("   Valid files read:", len(dfs), "| problematic:", bad)
assert len(dfs) > 0, "Could not read any valid CSV file."

data = pd.concat(dfs, ignore_index=True)

# Monthly spatial mean (one mean per month_date)
monthly = (data
           .groupby("month_date", as_index=False)["sst"]
           .mean()
           .sort_values("month_date"))

# Optional filter by MONTH (e.g., 7 = July across years)
months_en = {1:"January",2:"February",3:"March",4:"April",5:"May",6:"June",
             7:"July",8:"August",9:"September",10:"October",11:"November",12:"December"}
if MONTH is not None:
    monthly = monthly[monthly["month_date"].dt.month == MONTH].copy()
    title_suffix = f" ({months_en[MONTH]})"
    month_slug = f"{MONTH:02d}"
else:
    title_suffix = " (all months)"
    month_slug = "allmonths"

print("   Time span:", monthly["month_date"].min(), "→", monthly["month_date"].max())
print("   # of monthly points:", len(monthly))
assert len(monthly) > 0, "No data after the applied filter."

# --------- 4) Plot ----------
print("\n▶ Step 4/5: Plotting (NO smoothing, 2-year ticks)...")
plt.figure()
plt.plot(monthly["month_date"], monthly["sst"], linestyle="-")  # no smoothing
plt.title("SST - Monthly Mean" + title_suffix + " (X-axis ticks every 2 years) - MO/L3m")
plt.xlabel("Year")
plt.ylabel("SST (°C)")
plt.grid(True)

ax = plt.gca()
ax.xaxis.set_major_locator(YearLocator(base=2))    # ticks every 2 years
ax.xaxis.set_major_formatter(DateFormatter('%Y'))
plt.tight_layout()

# --------- 5) Save images ----------
print("\n▶ Step 5/5: Saving figures (PNG and SVG)...")
fig_dir = os.path.join(CSV_DIR, "fig")
os.makedirs(fig_dir, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
base_name = f"sst_monthly_mean_MO_L3m_{month_slug}_tick2years_{ts}"

png_path = os.path.join(fig_dir, base_name + ".png")
svg_path = os.path.join(fig_dir, base_name + ".svg")

plt.savefig(png_path, dpi=150)   # PNG
plt.savefig(svg_path)            # SVG
plt.show()

print("   ✓ PNG saved to:", png_path)
print("   ✓ SVG saved to:", svg_path)
print("\n✅ Done.")


In [ ]:
# Codigo 07 - ok =====================================================
# === SST Monthly Mean (NO smoothing) | X-axis ticks every 2 years | MO/L3m ===
MONTH = 12  # None = todos os meses; ou 1..12 para um mês específico (ex.: 7=Julho)

CSV_DIR = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados/MO/L3m/csv_export"

try:
    from google.colab import drive
    print("↪ Mounting Google Drive...")
    drive.mount('/content/drive')
except Exception:
    pass

import os, glob
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, DateFormatter
from datetime import datetime

assert os.path.isdir(CSV_DIR), f"Directory not found: {CSV_DIR}"
files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv.gz"))) + \
        sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
assert files, "No CSV files found."

def load_one_csv_monthly(path):
    head = pd.read_csv(path, nrows=0)
    cols = list(head.columns)
    sst_col = next((c for c in ["sst","sst4","sea_surface_temperature"] if c in cols), None)
    if sst_col is None:
        sst_like = [c for c in cols if "sst" in c.lower()]
        sst_col = sst_like[0] if sst_like else None
    if sst_col is None:
        return None

    if "date" in cols:
        df = pd.read_csv(path, usecols=["date", sst_col])
        dt = pd.to_datetime(df["date"], errors="coerce")
        month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        out = pd.DataFrame({"month_date": month_date, "sst": pd.to_numeric(df[sst_col], errors="coerce")})
    else:
        need = [c for c in ["year","month","datetime_iso"] if c in cols]
        if not need: return None
        dft = pd.read_csv(path, usecols=need)
        if {"year","month"}.issubset(dft.columns):
            y = pd.to_numeric(dft["year"], errors="coerce")
            m = pd.to_numeric(dft["month"], errors="coerce")
            month_date = pd.to_datetime(dict(year=y, month=m, day=1), errors="coerce")
        elif "datetime_iso" in dft.columns:
            dt = pd.to_datetime(dft["datetime_iso"], errors="coerce")
            month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        else:
            return None
        sst_series = pd.read_csv(path, usecols=[sst_col])[sst_col]
        out = pd.DataFrame({"month_date": month_date, "sst": pd.to_numeric(sst_series, errors="coerce")})

    out = out.dropna(subset=["month_date","sst"])
    out["month_date"] = pd.to_datetime(out["month_date"]).dt.normalize()
    return out

dfs = []
for f in files:
    dfx = load_one_csv_monthly(f)
    if dfx is not None and not dfx.empty:
        dfs.append(dfx)
assert dfs, "Could not read any valid CSV file."

data = pd.concat(dfs, ignore_index=True)

monthly = (data.groupby("month_date", as_index=False)["sst"]
           .mean()
           .sort_values("month_date"))

months_en = {1:"January",2:"February",3:"March",4:"April",5:"May",6:"June",
             7:"July",8:"August",9:"September",10:"October",11:"November",12:"December"}
if MONTH is not None:
    monthly = monthly[monthly["month_date"].dt.month == MONTH].copy()
    title_suffix = f" ({months_en[MONTH]})"
    month_slug = f"{MONTH:02d}"
else:
    title_suffix = " (all months)"
    month_slug = "allmonths"

assert not monthly.empty, "No data after the applied filter."

import matplotlib.pyplot as plt
plt.figure()
plt.plot(monthly["month_date"], monthly["sst"], linestyle="-", marker="o", markersize=2)  # ← pontos mensais
plt.title("SST - Monthly Mean" + title_suffix + " (X-axis ticks every 2 years) - MO/L3m")
plt.xlabel("Year")
plt.ylabel("SST (°C)")
plt.grid(True)
ax = plt.gca()
ax.xaxis.set_major_locator(YearLocator(base=2))
ax.xaxis.set_major_formatter(DateFormatter('%Y'))
plt.tight_layout()

fig_dir = os.path.join(CSV_DIR, "fig")
os.makedirs(fig_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
base = f"sst_monthly_mean_MO_L3m_{month_slug}_tick2years_{ts}"
plt.savefig(os.path.join(fig_dir, base + ".png"), dpi=150)
plt.savefig(os.path.join(fig_dir, base + ".svg"))
plt.show()
print("Saved to:", os.path.join(fig_dir, base + ".png"))


In [ ]:
# Codigo 08 - ok =====================================================
# === SST: one point per year = spatial mean for the chosen month | MO/L3m ===
# - MONTH is REQUIRED: 1..12 (1=Jan, 7=Jul, 12=Dec)
# - Reads all monthly CSVs under MO/L3m/csv_export
# - For the chosen MONTH, computes one spatial mean per year (→ one point per year)
# - Plots markers (no smoothing), X-axis ticks every 2 years
# - Saves PNG and SVG in csv_export/fig

# >>> Choose the month (1..12)
MONTH = 12    # e.g., 7 = July, 12 = December

# CSV directory (monthly L3m)
CSV_DIR = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados/MO/L3m/csv_export"

# --- Mount Google Drive (Colab) ---
try:
    from google.colab import drive
    print("↪ Mounting Google Drive...")
    drive.mount('/content/drive')
except Exception:
    print("↪ Non-Colab environment (skipping mount).")

import os, glob
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, DateFormatter
from datetime import datetime

# --- Checks ---
assert isinstance(MONTH, int) and 1 <= MONTH <= 12, "Set MONTH as an integer from 1 to 12."
assert os.path.isdir(CSV_DIR), f"Directory not found: {CSV_DIR}"

files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv.gz"))) + \
        sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
assert files, "No CSV files found."

# --- Loader for monthly CSVs ---
def load_one_csv_monthly(path):
    """
    Returns a DataFrame with:
      - month_date (timestamp normalized to 1st of month)
      - year (int), month (int)
      - sst (float)
    Works whether the CSV has 'date' or ('year','month') or 'datetime_iso'.
    """
    head = pd.read_csv(path, nrows=0)
    cols = list(head.columns)

    # Detect SST column
    sst_col = next((c for c in ["sst","sst4","sea_surface_temperature"] if c in cols), None)
    if sst_col is None:
        sst_like = [c for c in cols if "sst" in c.lower()]
        sst_col = sst_like[0] if sst_like else None
    if sst_col is None:
        return None

    # Build month_date/year/month
    if "date" in cols:
        df = pd.read_csv(path, usecols=["date", sst_col])
        dt = pd.to_datetime(df["date"], errors="coerce")
        month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        out = pd.DataFrame({
            "month_date": month_date,
            "year": month_date.dt.year,
            "month": month_date.dt.month,
            "sst": pd.to_numeric(df[sst_col], errors="coerce")
        })
    else:
        needed = [c for c in ["year","month","datetime_iso"] if c in cols]
        if not needed:
            return None
        if {"year","month"}.issubset(needed):
            dft = pd.read_csv(path, usecols=["year","month"])
            y = pd.to_numeric(dft["year"], errors="coerce")
            m = pd.to_numeric(dft["month"], errors="coerce")
            month_date = pd.to_datetime(dict(year=y, month=m, day=1), errors="coerce")
        elif "datetime_iso" in needed:
            dft = pd.read_csv(path, usecols=["datetime_iso"])
            dt = pd.to_datetime(dft["datetime_iso"], errors="coerce")
            month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        else:
            return None
        sst_series = pd.read_csv(path, usecols=[sst_col])[sst_col]
        out = pd.DataFrame({
            "month_date": month_date,
            "year": month_date.dt.year,
            "month": month_date.dt.month,
            "sst": pd.to_numeric(sst_series, errors="coerce")
        })

    out = out.dropna(subset=["month_date","year","month","sst"])
    out["year"] = out["year"].astype(int)
    out["month"] = out["month"].astype(int)
    out["month_date"] = pd.to_datetime(out["month_date"]).dt.normalize()
    return out[["month_date","year","month","sst"]]

# --- Read & stack ---
dfs = []
bad = 0
for f in files:
    try:
        dfx = load_one_csv_monthly(f)
        if dfx is not None and not dfx.empty:
            dfs.append(dfx)
    except Exception:
        bad += 1

assert dfs, "Could not read any valid monthly CSV."
data = pd.concat(dfs, ignore_index=True)

# --- Filter chosen month and compute 'one point per year' ---
month_data = data[data["month"] == MONTH].copy()
# Each file is monthly, but to be safe we compute spatial mean for each year-month,
# then group by year to get a single value per year (in case duplicates exist).
per_month_mean = (month_data.groupby(["year","month"], as_index=False)["sst"]
                  .mean())
annual_point_for_month = (per_month_mean.groupby("year", as_index=False)["sst"]
                          .mean()
                          .sort_values("year"))

assert not annual_point_for_month.empty, "No data for the selected MONTH in available CSVs."

# --- Plot (markers, no smoothing), 2-year ticks ---
months_en = {1:"January",2:"February",3:"March",4:"April",5:"May",6:"June",
             7:"July",8:"August",9:"September",10:"October",11:"November",12:"December"}
title_suffix = f" ({months_en[MONTH]} across years)"

# Use a datetime axis for nice tick formatting (Jan 1st of each year)
annual_point_for_month["year_date"] = pd.to_datetime(annual_point_for_month["year"].astype(str) + "-01-01")

plt.figure()
plt.plot(annual_point_for_month["year_date"], annual_point_for_month["sst"],
         linestyle="-", marker="o")
plt.title("SST - One point per year" + title_suffix + " - MO/L3m")
plt.xlabel("Year")
plt.ylabel("SST (°C)")
plt.grid(True)
ax = plt.gca()
ax.xaxis.set_major_locator(YearLocator(base=2))   # ticks every 2 years
ax.xaxis.set_major_formatter(DateFormatter('%Y'))
plt.tight_layout()

# --- Save figures ---
fig_dir = os.path.join(CSV_DIR, "fig")
os.makedirs(fig_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
base_name = f"sst_one_point_per_year_month_{MONTH:02d}_MO_L3m_tick2years_{ts}"

png_path = os.path.join(fig_dir, base_name + ".png")
svg_path = os.path.join(fig_dir, base_name + ".svg")
plt.savefig(png_path, dpi=150)
plt.savefig(svg_path)
plt.show()

print("Saved:")
print("  PNG:", png_path)
print("  SVG:", svg_path)


In [ ]:
# Codigo 09 - ok =====================================================
# SST - One point per year (chosen month) with linear regression | MO/L3m
# - Filters a chosen MONTH (1..12), computes one spatial mean per year
# - Fits linear regression y = a*x + b (x = year)
# - Plots data + regression (NO smoothing), X-axis ticks every 2 years
# - Saves PNG/SVG + CSV with data & fitted values

# === Choose the month (1..12) ===
MONTH = 7   # e.g., 7 = July, 12 = December

# === CSV directory (monthly L3m) ===
CSV_DIR = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados/MO/L3m/csv_export"

# --- Mount Google Drive (Colab) ---
from google.colab import drive
print("↪ Mounting Google Drive...")
drive.mount('/content/drive')

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, DateFormatter
from datetime import datetime

# 1) Check directory
print("\n▶ Step 1/7: Checking directory...")
assert os.path.isdir(CSV_DIR), f"Directory not found: {CSV_DIR}"

# 2) List files
print("\n▶ Step 2/7: Listing .csv.gz/.csv files...")
files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv.gz"))) + \
        sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
print("   Files found:", len(files))
assert files, "No CSV files found."

# Loader
def load_one_csv_monthly(path):
    head = pd.read_csv(path, nrows=0)
    cols = list(head.columns)
    sst_col = next((c for c in ["sst","sst4","sea_surface_temperature"] if c in cols), None)
    if sst_col is None:
        sst_like = [c for c in cols if "sst" in c.lower()]
        sst_col = sst_like[0] if sst_like else None
    if sst_col is None:
        return None

    if "date" in cols:
        df = pd.read_csv(path, usecols=["date", sst_col])
        dt = pd.to_datetime(df["date"], errors="coerce")
        month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        out = pd.DataFrame({
            "month_date": month_date,
            "year": month_date.dt.year,
            "month": month_date.dt.month,
            "sst": pd.to_numeric(df[sst_col], errors="coerce")
        })
    else:
        needed = [c for c in ["year","month","datetime_iso"] if c in cols]
        if not needed:
            return None
        if {"year","month"}.issubset(needed):
            dft = pd.read_csv(path, usecols=["year","month"])
            y = pd.to_numeric(dft["year"], errors="coerce")
            m = pd.to_numeric(dft["month"], errors="coerce")
            month_date = pd.to_datetime(dict(year=y, month=m, day=1), errors="coerce")
        elif "datetime_iso" in needed:
            dft = pd.read_csv(path, usecols=["datetime_iso"])
            dt = pd.to_datetime(dft["datetime_iso"], errors="coerce")
            month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        else:
            return None
        sst_series = pd.read_csv(path, usecols=[sst_col])[sst_col]
        out = pd.DataFrame({
            "month_date": month_date,
            "year": month_date.dt.year,
            "month": month_date.dt.month,
            "sst": pd.to_numeric(sst_series, errors="coerce")
        })

    out = out.dropna(subset=["month_date","year","month","sst"])
    out["year"] = out["year"].astype(int)
    out["month"] = out["month"].astype(int)
    out["month_date"] = pd.to_datetime(out["month_date"]).dt.normalize()
    return out[["month_date","year","month","sst"]]

# 3) Read & stack
print("\n▶ Step 3/7: Reading and stacking data...")
dfs, bad = [], 0
for f in files:
    try:
        dfx = load_one_csv_monthly(f)
        if dfx is not None and not dfx.empty:
            dfs.append(dfx)
    except Exception:
        bad += 1
print("   Valid files read:", len(dfs), "| problematic:", bad)
assert dfs, "Could not read any valid monthly CSV."
data = pd.concat(dfs, ignore_index=True)

# 4) Filter month and build per-year series
print("\n▶ Step 4/7: Building one-point-per-year series...")
assert 1 <= MONTH <= 12, "MONTH must be 1..12"
month_data = data[data["month"] == MONTH].copy()
per_month_mean = (month_data.groupby(["year","month"], as_index=False)["sst"].mean())
annual_series = (per_month_mean.groupby("year", as_index=False)["sst"].mean().sort_values("year"))
assert not annual_series.empty, "No data for the selected MONTH."

print("   Years:", int(annual_series['year'].min()), "→", int(annual_series['year'].max()))
print("   Points:", len(annual_series))

# 5) Linear regression
print("\n▶ Step 5/7: Linear regression y = a*x + b (x = year)...")
x = annual_series["year"].astype(float).values
y = annual_series["sst"].astype(float).values
a, b = np.polyfit(x, y, 1)   # slope (°C/year), intercept (°C)
y_hat = a * x + b
r = np.corrcoef(x, y)[0,1] if len(x) > 1 else np.nan
r2 = r**2 if np.isfinite(r) else np.nan
annual_series["y_fit"] = y_hat

print(f"   Slope (a): {a:.6f} °C/year")
print(f"   Intercept (b): {b:.6f} °C")
print(f"   R^2: {r2:.6f}")

# 6) Plot
print("\n▶ Step 6/7: Plotting data + regression (NO smoothing)...")
annual_series["year_date"] = pd.to_datetime(annual_series["year"].astype(str) + "-01-01")
plt.figure()
plt.plot(annual_series["year_date"], annual_series["sst"], linestyle="", marker="o", label="Data (annual point)")
plt.plot(annual_series["year_date"], annual_series["y_fit"], linestyle="-", label="Linear fit")
plt.title(f"SST - One point per year (month={MONTH:02d}) with linear fit - MO/L3m")
plt.xlabel("Year")
plt.ylabel("SST (°C)")
plt.grid(True)
plt.legend()
ax = plt.gca()
ax.xaxis.set_major_locator(YearLocator(base=2))
ax.xaxis.set_major_formatter(DateFormatter('%Y'))
plt.tight_layout()

# 7) Save outputs
fig_dir = os.path.join(CSV_DIR, "fig")
os.makedirs(fig_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
base = f"sst_one_point_per_year_month_{MONTH:02d}_with_linfit_MO_L3m_{ts}"
png = os.path.join(fig_dir, base + ".png")
svg = os.path.join(fig_dir, base + ".svg")
csv = os.path.join(fig_dir, base + ".csv")

plt.savefig(png, dpi=150)
plt.savefig(svg)
plt.show()
annual_series.to_csv(csv, index=False)

print("\n▶ Linear Regression Parameters")
print(f"   Equation: SST = {a:.6f} * Year + {b:.6f}")
print(f"   Slope a (°C/year): {a:.6f}")
print(f"   Intercept b (°C):  {b:.6f}")
print(f"   R^2: {r2:.6f}")

print("\n▶ Saved files:")
print("   PNG:", png)
print("   SVG:", svg)
print("   CSV:", csv)


In [ ]:
# Codigo 10 - ok =====================================================
# === SST — Pontos de dezembro vs retas (2002–2025) | MO/L3m (sem caas_jupyter_tools) ===
# Lê MO/L3m/csv_export, monta a tabela (artigo vs sua reta de dezembro + pontos observados),
# plota pontos de dezembro + ambas as retas, e salva PNG/SVG + CSV.

# ---- Caminho dos CSVs (mensal L3m) ----
CSV_DIR = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados/MO/L3m/csv_export"

# ---- Intervalo de anos ----
YEAR_START, YEAR_END = 2002, 2025
MONTH = 12  # dezembro

# ---- Coeficientes das retas ----
# Artigo (Figura 3): y = 0.0502*x - 71.863
a_art, b_art = 0.0502, -71.863
# Sua regressão (dezembro): y = 0.024035*x - 19.068216
a_dec, b_dec = 0.024035, -19.068216

# --- Montar Drive (Colab) ---
try:
    from google.colab import drive
    print("↪ Montando Google Drive...")
    drive.mount('/content/drive')
except Exception as e:
    print("↪ Ambiente fora do Colab (pular montagem):", e)

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, DateFormatter
from datetime import datetime

# 1) Conferir pasta
assert os.path.isdir(CSV_DIR), f"Diretório não encontrado: {CSV_DIR}"

# 2) Listar CSVs
files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv.gz"))) + \
        sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
assert files, "Nenhum CSV encontrado."

# Loader mensal
def load_one_csv_monthly(path):
    head = pd.read_csv(path, nrows=0)
    cols = list(head.columns)
    sst_col = next((c for c in ["sst","sst4","sea_surface_temperature"] if c in cols), None)
    if sst_col is None:
        sst_like = [c for c in cols if "sst" in c.lower()]
        sst_col = sst_like[0] if sst_like else None
    if sst_col is None:
        return None

    if "date" in cols:
        df = pd.read_csv(path, usecols=["date", sst_col])
        dt = pd.to_datetime(df["date"], errors="coerce")
        month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        out = pd.DataFrame({
            "month_date": month_date,
            "year": month_date.dt.year,
            "month": month_date.dt.month,
            "sst": pd.to_numeric(df[sst_col], errors="coerce")
        })
    else:
        needed = [c for c in ["year","month","datetime_iso"] if c in cols]
        if not needed:
            return None
        if {"year","month"}.issubset(needed):
            dft = pd.read_csv(path, usecols=["year","month"])
            y = pd.to_numeric(dft["year"], errors="coerce")
            m = pd.to_numeric(dft["month"], errors="coerce")
            month_date = pd.to_datetime(dict(year=y, month=m, day=1), errors="coerce")
        elif "datetime_iso" in needed:
            dft = pd.read_csv(path, usecols=["datetime_iso"])
            dt = pd.to_datetime(dft["datetime_iso"], errors="coerce")
            month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        else:
            return None
        sst_series = pd.read_csv(path, usecols=[sst_col])[sst_col]
        out = pd.DataFrame({
            "month_date": month_date,
            "year": month_date.dt.year,
            "month": month_date.dt.month,
            "sst": pd.to_numeric(sst_series, errors="coerce")
        })

    out = out.dropna(subset=["month_date","year","month","sst"])
    out["year"] = out["year"].astype(int)
    out["month"] = out["month"].astype(int)
    out["month_date"] = pd.to_datetime(out["month_date"]).dt.normalize()
    return out[["month_date","year","month","sst"]]

# 3) Ler & empilhar
dfs = []
for f in files:
    try:
        dfx = load_one_csv_monthly(f)
        if dfx is not None and not dfx.empty:
            dfs.append(dfx)
    except Exception:
        pass
assert dfs, "Falha ao ler CSVs mensais."
data = pd.concat(dfs, ignore_index=True)

# 4) Tabela 2002–2025 (artigo vs dezembro) + pontos observados de dezembro
years = np.arange(YEAR_START, YEAR_END + 1, dtype=int)
y_article  = a_art*years + b_art
y_december = a_dec*years + b_dec
tbl = pd.DataFrame({
    "year": years,
    "sst_article_fit_degC": y_article,
    "sst_december_fit_degC": y_december,
    "diff_december_minus_article_degC": y_december - y_article
})

# Pontos observados (média espacial de dezembro → 1 valor por ano)
dec_data = data[data["month"] == 12].copy()
per_month_mean = (dec_data.groupby(["year","month"], as_index=False)["sst"].mean())
dec_points = (per_month_mean.groupby("year", as_index=False)["sst"].mean()
              .sort_values("year"))
tbl = tbl.merge(dec_points.rename(columns={"sst":"sst_december_point_degC"}),
                on="year", how="left")

# Mostrar primeiras linhas
print("\nVista prévia da tabela (2002–2025):")
print(tbl.head(12).round(4).to_string(index=False))

# Salvar CSV
csv_out = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados/MO/L3m/csv_export/fig/sst_fits_and_december_points_2002_2025.csv"
os.makedirs(os.path.dirname(csv_out), exist_ok=True)
tbl.to_csv(csv_out, index=False)
print("\n✓ CSV salvo em:", csv_out)

# 5) Gráfico: pontos de dezembro + duas retas
date_axis = pd.to_datetime(tbl["year"].astype(str) + "-01-01")
plt.figure()
# pontos
if not dec_points.empty:
    dec_dates = pd.to_datetime(dec_points["year"].astype(str) + "-01-01")
    plt.plot(dec_dates, dec_points["sst"], linestyle="", marker="o", label="Ponto observado (dezembro)")
# retas
plt.plot(date_axis, tbl["sst_december_fit_degC"], linestyle="-", label="Reta (dezembro)")
plt.plot(date_axis, tbl["sst_article_fit_degC"], linestyle="-", label="Reta (artigo)")
plt.title("SST — Pontos de dezembro vs retas (2002–2025)")
plt.xlabel("Ano")
plt.ylabel("SST (°C)")
plt.grid(True)
ax = plt.gca()
ax.xaxis.set_major_locator(YearLocator(base=2))
ax.xaxis.set_major_formatter(DateFormatter('%Y'))
plt.legend()
plt.tight_layout()

# Salvar figura
fig_dir = os.path.join(CSV_DIR, "fig")
os.makedirs(fig_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
png_path = os.path.join(fig_dir, f"sst_dezembro_pontos_vs_retas_2002_2025_{ts}.png")
svg_path = os.path.join(fig_dir, f"sst_dezembro_pontos_vs_retas_2002_2025_{ts}.svg")
plt.savefig(png_path, dpi=150)
plt.savefig(svg_path)
plt.show()
print("✓ PNG:", png_path)
print("✓ SVG:", svg_path)


In [ ]:
# Codigo 11 - ok =====================================================
# === SST — Pontos de dezembro vs retas (2002–2025) | MO/L3m (com curva dos pontos médios) ===

CSV_DIR = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados/MO/L3m/csv_export"
YEAR_START, YEAR_END = 2002, 2025

# Retas:
a_art, b_art = 0.0502, -71.863               # Artigo (Fig. 3)
a_dec, b_dec = 0.024035, -19.068216          # Sua regressão para dezembro

# --- Montar Drive (Colab) ---
try:
    from google.colab import drive
    print("↪ Montando Google Drive...")
    drive.mount('/content/drive')
except Exception:
    pass

import os, glob
import numpy as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, DateFormatter
from datetime import datetime

# 1) Conferir pasta e arquivos
assert os.path.isdir(CSV_DIR), f"Diretório não encontrado: {CSV_DIR}"
files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv.gz"))) + \
        sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
assert files, "Nenhum CSV encontrado."

# Loader mensal
def load_one_csv_monthly(path):
    head = pd.read_csv(path, nrows=0)
    cols = list(head.columns)
    sst_col = next((c for c in ["sst","sst4","sea_surface_temperature"] if c in cols), None)
    if sst_col is None:
        sst_like = [c for c in cols if "sst" in c.lower()]
        sst_col = sst_like[0] if sst_like else None
    if sst_col is None:
        return None

    if "date" in cols:
        df = pd.read_csv(path, usecols=["date", sst_col])
        dt = pd.to_datetime(df["date"], errors="coerce")
        month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        out = pd.DataFrame({
            "month_date": month_date,
            "year": month_date.dt.year,
            "month": month_date.dt.month,
            "sst": pd.to_numeric(df[sst_col], errors="coerce")
        })
    else:
        need = [c for c in ["year","month","datetime_iso"] if c in cols]
        if not need: return None
        if {"year","month"}.issubset(need):
            dft = pd.read_csv(path, usecols=["year","month"])
            y = pd.to_numeric(dft["year"], errors="coerce")
            m = pd.to_numeric(dft["month"], errors="coerce")
            month_date = pd.to_datetime(dict(year=y, month=m, day=1), errors="coerce")
        elif "datetime_iso" in need:
            dft = pd.read_csv(path, usecols=["datetime_iso"])
            dt = pd.to_datetime(dft["datetime_iso"], errors="coerce")
            month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        else:
            return None
        sst_series = pd.read_csv(path, usecols=[sst_col])[sst_col]
        out = pd.DataFrame({
            "month_date": month_date,
            "year": month_date.dt.year,
            "month": month_date.dt.month,
            "sst": pd.to_numeric(sst_series, errors="coerce")
        })

    out = out.dropna(subset=["month_date","year","month","sst"])
    out["year"] = out["year"].astype(int)
    out["month"] = out["month"].astype(int)
    out["month_date"] = pd.to_datetime(out["month_date"]).dt.normalize()
    return out[["month_date","year","month","sst"]]

# 2) Ler & empilhar
dfs = []
for f in files:
    try:
        dfx = load_one_csv_monthly(f)
        if dfx is not None and not dfx.empty:
            dfs.append(dfx)
    except Exception:
        pass
assert dfs, "Falha ao ler CSVs mensais."
data = pd.concat(dfs, ignore_index=True)

# 3) Tabela 2002–2025 com as duas retas
years = np.arange(YEAR_START, YEAR_END + 1, dtype=int)
y_article  = a_art*years + b_art
y_december = a_dec*years + b_dec
tbl = pd.DataFrame({
    "year": years,
    "sst_article_fit_degC": y_article,
    "sst_december_fit_degC": y_december,
    "diff_december_minus_article_degC": y_december - y_article
})

# 4) Pontos observados de dezembro: média espacial -> 1 valor por ano
dec = data[data["month"] == 12].copy()
per_month = dec.groupby(["year","month"], as_index=False)["sst"].mean()
dec_points = per_month.groupby("year", as_index=False)["sst"].mean().sort_values("year")

# merge (opcional, caso queira ter na mesma tabela)
tbl = tbl.merge(dec_points.rename(columns={"sst":"sst_december_point_degC"}), on="year", how="left")

# 5) Plot: **curva dos pontos médios de dezembro** + retas
date_axis = pd.to_datetime(tbl["year"].astype(str) + "-01-01")
plt.figure()

# *** Aqui está a curva dos pontos médios de dezembro ***
if not dec_points.empty:
    dec_dates = pd.to_datetime(dec_points["year"].astype(str) + "-01-01")
    plt.plot(dec_dates, dec_points["sst"], linestyle="-", marker="o",
             label="December mean (observed)")

# Retas (artigo e sua de dezembro)
plt.plot(date_axis, tbl["sst_december_fit_degC"], linestyle="-", label="Linear fit (December)")
plt.plot(date_axis, tbl["sst_article_fit_degC"], linestyle="-", label="Linear fit (Article)")

plt.title("SST — December mean curve vs linear fits (2002–2025)")
plt.xlabel("Year")
plt.ylabel("SST (°C)")
plt.grid(True)
ax = plt.gca()
ax.xaxis.set_major_locator(YearLocator(base=2))
ax.xaxis.set_major_formatter(DateFormatter('%Y'))
plt.legend()
plt.tight_layout()

# 6) Salvar imagem + CSV da tabela
fig_dir = os.path.join(CSV_DIR, "fig")
os.makedirs(fig_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
png_path = os.path.join(fig_dir, f"sst_december_curve_vs_fits_2002_2025_{ts}.png")
svg_path = os.path.join(fig_dir, f"sst_december_curve_vs_fits_2002_2025_{ts}.svg")
plt.savefig(png_path, dpi=150)
plt.savefig(svg_path)
plt.show()

csv_out = os.path.join(fig_dir, "sst_fits_and_december_points_2002_2025.csv")
tbl.to_csv(csv_out, index=False)

print("✓ PNG:", png_path)
print("✓ SVG:", svg_path)
print("✓ CSV:", csv_out)


In [ ]:
# Codigo 12 - ok =====================================================
# === SST — July (MONTH=7): curva (linha) ligando os pontos médios anuais + regressão | MO/L3m ===
# - Lê CSVs mensais em MO/L3m/csv_export
# - Calcula 1 valor por ano para JULHO (média espacial daquele mês)
# - Plota a CURVA dos pontos anuais (linha+marcadores) e a reta de regressão
# - Salva PNG/SVG e CSV com a série e y_fit

MONTH = 7  # julho
CSV_DIR = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados/MO/L3m/csv_export"

# --- Montar Google Drive (Colab) ---
try:
    from google.colab import drive
    print("↪ Montando Google Drive...")
    drive.mount('/content/drive')
except Exception:
    print("↪ Ambiente fora do Colab (pular montagem).")

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, DateFormatter
from datetime import datetime

# 1) Conferências básicas
assert os.path.isdir(CSV_DIR), f"Diretório não encontrado: {CSV_DIR}"
files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv.gz"))) + \
        sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
assert files, "Nenhum CSV encontrado."

# 2) Loader mensal -> DataFrame com month_date, year, month, sst
def load_one_csv_monthly(path):
    head = pd.read_csv(path, nrows=0)
    cols = list(head.columns)

    sst_col = next((c for c in ["sst","sst4","sea_surface_temperature"] if c in cols), None)
    if sst_col is None:
        sst_like = [c for c in cols if "sst" in c.lower()]
        sst_col = sst_like[0] if sst_like else None
    if sst_col is None:
        return None

    if "date" in cols:
        df = pd.read_csv(path, usecols=["date", sst_col])
        dt = pd.to_datetime(df["date"], errors="coerce")
        month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        out = pd.DataFrame({
            "month_date": month_date,
            "year": month_date.dt.year,
            "month": month_date.dt.month,
            "sst": pd.to_numeric(df[sst_col], errors="coerce")
        })
    else:
        needed = [c for c in ["year","month","datetime_iso"] if c in cols]
        if not needed:
            return None
        if {"year","month"}.issubset(needed):
            dft = pd.read_csv(path, usecols=["year","month"])
            y = pd.to_numeric(dft["year"], errors="coerce")
            m = pd.to_numeric(dft["month"], errors="coerce")
            month_date = pd.to_datetime(dict(year=y, month=m, day=1), errors="coerce")
        elif "datetime_iso" in needed:
            dft = pd.read_csv(path, usecols=["datetime_iso"])
            dt = pd.to_datetime(dft["datetime_iso"], errors="coerce")
            month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        else:
            return None
        sst_series = pd.read_csv(path, usecols=[sst_col])[sst_col]
        out = pd.DataFrame({
            "month_date": month_date,
            "year": month_date.dt.year,
            "month": month_date.dt.month,
            "sst": pd.to_numeric(sst_series, errors="coerce")
        })

    out = out.dropna(subset=["month_date","year","month","sst"])
    out["year"] = out["year"].astype(int)
    out["month"] = out["month"].astype(int)
    out["month_date"] = pd.to_datetime(out["month_date"]).dt.normalize()
    return out[["month_date","year","month","sst"]]

# 3) Ler e empilhar
dfs = []
for f in files:
    try:
        dfx = load_one_csv_monthly(f)
        if dfx is not None and not dfx.empty:
            dfs.append(dfx)
    except Exception:
        pass
assert dfs, "Falha ao ler CSVs mensais."
data = pd.concat(dfs, ignore_index=True)

# 4) Filtrar JULHO e criar série: 1 ponto por ano (média espacial daquele mês)
month_data = data[data["month"] == MONTH].copy()
per_month_mean = month_data.groupby(["year","month"], as_index=False)["sst"].mean()
annual_series = per_month_mean.groupby("year", as_index=False)["sst"].mean().sort_values("year")
assert not annual_series.empty, "Não há dados para JULHO nos CSVs."

# 5) Regressão linear y = a*x + b (x=ano)
x = annual_series["year"].astype(float).values
y = annual_series["sst"].astype(float).values
a, b = np.polyfit(x, y, 1)                    # slope/intercept
annual_series["y_fit"] = a * x + b

print(f"Slope (°C/ano): {a:.6f}  |  Intercept (°C): {b:.6f}")

# 6) Plot — **curva dos pontos médios de JULHO** (linha+marcador) + reta de regressão
annual_series["year_date"] = pd.to_datetime(annual_series["year"].astype(str) + "-01-01")

plt.figure()
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# AQUI ESTÁ A CURVA (linha) LIGANDO OS PONTOS DE JULHO:
plt.plot(annual_series["year_date"], annual_series["sst"],
         linestyle="-", marker="o", label="July mean (observed)")
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

# Reta de regressão
plt.plot(annual_series["year_date"], annual_series["y_fit"],
         linestyle="-", label=f"Linear fit (a={a:.3f} °C/yr)")

plt.title("SST — July annual means (curve) + linear fit - MO/L3m")
plt.xlabel("Year")
plt.ylabel("SST (°C)")
plt.grid(True)
ax = plt.gca()
ax.xaxis.set_major_locator(YearLocator(base=2))
ax.xaxis.set_major_formatter(DateFormatter('%Y'))
plt.legend()
plt.tight_layout()

# 7) Salvar figura e CSV
fig_dir = os.path.join(CSV_DIR, "fig")
os.makedirs(fig_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
base = f"sst_july_curve_with_linfit_MO_L3m_{ts}"
png = os.path.join(fig_dir, base + ".png")
svg = os.path.join(fig_dir, base + ".svg")
csv = os.path.join(fig_dir, base + ".csv")

plt.savefig(png, dpi=150)
plt.savefig(svg)
plt.show()

annual_series.to_csv(csv, index=False)
print("Salvos:")
print(" PNG:", png)
print(" SVG:", svg)
print(" CSV:", csv)


In [ ]:
# Codigo 13 - =====================================================
# === SST July — Uncertainty quantification for the 2025 projection | MO/L3m ===
# This cell is the response to the reviewers. It reproduces, from the same
# localized MODIS L3m July time series used in Codigo 12, the following:
#   (1) OLS slope significance via t-test (with 95% CI for the slope)
#   (2) 95% confidence interval for the mean ŷ(2025)
#   (3) 95% prediction interval for a single observed value in July 2025
#   (4) Probability of exceeding decision thresholds in July 2025
#       (P(SST>29.5°C), P(SST>30°C), P(SST>30.5°C), P(SST>31°C))
#   (5) Mann–Kendall non-parametric trend test + Theil–Sen slope estimator
#
# The numbers produced here are the ones reported in Annex I, Sections
# 1.4.1, 1.4.2, and 1.4.3 of the revised manuscript.

MONTH    = 7      # July
TARGET_Y = 2025   # year to project to
CSV_DIR  = "/content/drive/MyDrive/Colab Notebooks/SST_Singapore_Strait/Dados/MO/L3m/csv_export"

# --- Mount Google Drive (Colab) ---
try:
    from google.colab import drive
    print("↪ Mounting Google Drive...")
    drive.mount('/content/drive')
except Exception:
    print("↪ Non-Colab environment (skipping mount).")

import os, glob
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, DateFormatter
from datetime import datetime

# -------------------------------------------------------------------------
# 1) Load monthly CSVs (same loader logic as Codigo 06/07/08/09/12)
# -------------------------------------------------------------------------
assert os.path.isdir(CSV_DIR), f"Directory not found: {CSV_DIR}"
files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv.gz"))) + \
        sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
assert files, "No CSV files found."

def load_one_csv_monthly(path):
    head = pd.read_csv(path, nrows=0)
    cols = list(head.columns)
    sst_col = next((c for c in ["sst","sst4","sea_surface_temperature"] if c in cols), None)
    if sst_col is None:
        sst_like = [c for c in cols if "sst" in c.lower()]
        sst_col = sst_like[0] if sst_like else None
    if sst_col is None:
        return None
    if "date" in cols:
        df = pd.read_csv(path, usecols=["date", sst_col])
        dt = pd.to_datetime(df["date"], errors="coerce")
        month_date = pd.to_datetime(dt.dt.to_period("M").dt.to_timestamp())
        out = pd.DataFrame({
            "year": month_date.dt.year,
            "month": month_date.dt.month,
            "sst": pd.to_numeric(df[sst_col], errors="coerce")
        })
    else:
        needed = [c for c in ["year","month","datetime_iso"] if c in cols]
        if not needed: return None
        if {"year","month"}.issubset(needed):
            dft = pd.read_csv(path, usecols=["year","month"])
            y = pd.to_numeric(dft["year"], errors="coerce")
            m = pd.to_numeric(dft["month"], errors="coerce")
            out = pd.DataFrame({"year": y, "month": m})
        else:
            dft = pd.read_csv(path, usecols=["datetime_iso"])
            dt = pd.to_datetime(dft["datetime_iso"], errors="coerce")
            out = pd.DataFrame({"year": dt.dt.year, "month": dt.dt.month})
        sst_series = pd.read_csv(path, usecols=[sst_col])[sst_col]
        out["sst"] = pd.to_numeric(sst_series, errors="coerce")
    out = out.dropna(subset=["year","month","sst"]).astype({"year": int, "month": int})
    return out[["year","month","sst"]]

dfs = []
for f in files:
    try:
        dfx = load_one_csv_monthly(f)
        if dfx is not None and not dfx.empty:
            dfs.append(dfx)
    except Exception:
        pass
assert dfs, "Failed to load monthly CSVs."
data = pd.concat(dfs, ignore_index=True)

# -------------------------------------------------------------------------
# 2) Build the July one-point-per-year series
# -------------------------------------------------------------------------
month_data    = data[data["month"] == MONTH].copy()
per_month     = month_data.groupby(["year","month"], as_index=False)["sst"].mean()
annual_series = per_month.groupby("year", as_index=False)["sst"].mean().sort_values("year")
annual_series = annual_series.reset_index(drop=True)

x = annual_series["year"].astype(float).values
y = annual_series["sst"].astype(float).values
n = len(x)
assert n >= 5, "Not enough July years to fit a trend."

print(f"\n▶ July series: n = {n}  ({int(x.min())}–{int(x.max())})")

# -------------------------------------------------------------------------
# 3) OLS regression with full uncertainty machinery
# -------------------------------------------------------------------------
# slope, intercept, R² from polyfit (matches Codigo 12 exactly)
a, b = np.polyfit(x, y, 1)
y_hat = a * x + b
ss_res = np.sum((y - y_hat) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1.0 - ss_res / ss_tot

# residual standard error (RMSE with n-2 degrees of freedom)
sigma_res = np.sqrt(ss_res / (n - 2))

# Sxx and standard errors of a, b
Sxx = np.sum((x - x.mean()) ** 2)
SE_a = sigma_res / np.sqrt(Sxx)
SE_b = sigma_res * np.sqrt(1.0 / n + x.mean() ** 2 / Sxx)

# t-test on the slope
t_a   = a / SE_a
p_a   = 2 * (1 - stats.t.cdf(abs(t_a), df=n - 2))
tcrit = stats.t.ppf(0.975, df=n - 2)
CI_a  = (a - tcrit * SE_a, a + tcrit * SE_a)

print("\n=== OLS regression for July ===")
print(f"  slope  a  : {a:.6f} °C/year")
print(f"  intercept b: {b:.4f} °C  (algebraic artefact at x=0)")
print(f"  R²        : {r2:.4f}")
print(f"  σ_res     : {sigma_res:.4f} °C")
print(f"  SE(a)     : {SE_a:.6f}    t={t_a:.3f}    p={p_a:.4f}    df={n-2}")
print(f"  95% CI(a) : [{CI_a[0]:.5f}, {CI_a[1]:.5f}] °C/year")

# -------------------------------------------------------------------------
# 4) Projection for TARGET_Y with confidence and prediction intervals
# -------------------------------------------------------------------------
x_new = float(TARGET_Y)
y_new = a * x_new + b

# SE of the predicted mean (confidence interval band)
SE_yhat = sigma_res * np.sqrt(1.0 / n + (x_new - x.mean()) ** 2 / Sxx)
CI_yhat = (y_new - tcrit * SE_yhat, y_new + tcrit * SE_yhat)

# SE of a single new observation (prediction interval)
SE_pred = sigma_res * np.sqrt(1.0 + 1.0 / n + (x_new - x.mean()) ** 2 / Sxx)
PI_obs  = (y_new - tcrit * SE_pred, y_new + tcrit * SE_pred)

print(f"\n=== Projection for July {TARGET_Y} ===")
print(f"  point estimate ŷ        : {y_new:.3f} °C")
print(f"  95% CI for the mean     : [{CI_yhat[0]:.3f}, {CI_yhat[1]:.3f}] °C")
print(f"  95% prediction interval : [{PI_obs[0]:.3f}, {PI_obs[1]:.3f}] °C  (USE THIS for event-risk talk)")

# -------------------------------------------------------------------------
# 5) Threshold exceedance probabilities under the model (Gaussian residual
#    assumption around the fitted line, with prediction-interval std error)
# -------------------------------------------------------------------------
def p_exceed(threshold, mu, sigma):
    return 1.0 - stats.norm.cdf(threshold, loc=mu, scale=sigma)

thresholds = [29.0, 29.5, 30.0, 30.5, 31.0]
print(f"\n=== Threshold exceedance probabilities for July {TARGET_Y} ===")
for th in thresholds:
    p = p_exceed(th, y_new, SE_pred)
    marker = ""
    if th == 30.0: marker = "  (event temperature recorded in 2025)"
    if th == 31.0: marker = "  (World Aquatics operational threshold)"
    print(f"  P(SST > {th:.1f} °C) = {p:.3f}  ({p*100:.1f}%){marker}")

# -------------------------------------------------------------------------
# 6) Mann–Kendall non-parametric trend test + Theil–Sen slope
# -------------------------------------------------------------------------
S = 0
for i in range(n - 1):
    S += np.sum(np.sign(y[i + 1:] - y[i]))
var_S = n * (n - 1) * (2 * n + 5) / 18.0
if S > 0:   Z = (S - 1) / np.sqrt(var_S)
elif S < 0: Z = (S + 1) / np.sqrt(var_S)
else:       Z = 0.0
p_MK = 2 * (1 - stats.norm.cdf(abs(Z)))

# Theil–Sen slope
slopes_all = []
for i in range(n - 1):
    slopes_all.extend((y[i + 1:] - y[i]) / (x[i + 1:] - x[i]))
sen_slope = np.median(slopes_all)

print(f"\n=== Non-parametric robustness check ===")
print(f"  Mann–Kendall S       : {int(S)}")
print(f"  Mann–Kendall Z       : {Z:.3f}")
print(f"  Mann–Kendall p-value : {p_MK:.4f}  (two-sided)")
print(f"  Theil–Sen slope      : {sen_slope:.6f} °C/year")

# -------------------------------------------------------------------------
# 7) Plot: data + fit + 95% CI band + 95% PI band + projection point
# -------------------------------------------------------------------------
x_grid = np.linspace(x.min(), TARGET_Y, 200)
y_grid = a * x_grid + b
se_y_grid = sigma_res * np.sqrt(1.0 / n + (x_grid - x.mean()) ** 2 / Sxx)
se_p_grid = sigma_res * np.sqrt(1.0 + 1.0 / n + (x_grid - x.mean()) ** 2 / Sxx)

x_grid_date = pd.to_datetime(pd.Series(x_grid).round().astype(int).astype(str) + "-01-01")
x_data_date = pd.to_datetime(annual_series["year"].astype(str) + "-01-01")
x_pred_date = pd.to_datetime(f"{TARGET_Y}-01-01")

plt.figure(figsize=(10, 5.5))
plt.fill_between(x_grid_date, y_grid - tcrit * se_p_grid, y_grid + tcrit * se_p_grid,
                 alpha=0.15, label="95% prediction interval")
plt.fill_between(x_grid_date, y_grid - tcrit * se_y_grid, y_grid + tcrit * se_y_grid,
                 alpha=0.30, label="95% CI of the mean")
plt.plot(x_grid_date, y_grid, linewidth=1.5, label=f"OLS fit (a = {a:.4f} °C/yr)")
plt.plot(x_data_date, y, "o", label="July annual means (observed)")
plt.plot(x_pred_date, y_new, "*", markersize=14,
         label=f"July {TARGET_Y}: {y_new:.2f} °C  (PI 95%: {PI_obs[0]:.2f}–{PI_obs[1]:.2f} °C)")

# Reference lines
plt.axhline(31.0, linestyle="--", linewidth=1, alpha=0.6,
            label="World Aquatics threshold (31 °C)")
plt.axhline(30.0, linestyle=":",  linewidth=1, alpha=0.6,
            label="Event temperature recorded (30 °C)")

plt.title(f"July SST in the Singapore Strait — OLS fit, 95% CI and 95% PI, projection to {TARGET_Y}")
plt.xlabel("Year")
plt.ylabel("July SST (°C)")
plt.grid(True, alpha=0.4)
ax = plt.gca()
ax.xaxis.set_major_locator(YearLocator(base=2))
ax.xaxis.set_major_formatter(DateFormatter("%Y"))
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()

fig_dir = os.path.join(CSV_DIR, "fig")
os.makedirs(fig_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
base = f"sst_july_uncertainty_projection_{TARGET_Y}_{ts}"
png = os.path.join(fig_dir, base + ".png")
svg = os.path.join(fig_dir, base + ".svg")
plt.savefig(png, dpi=150)
plt.savefig(svg)
plt.show()

# -------------------------------------------------------------------------
# 8) Export a tidy CSV with everything the revised Annex I cites
# -------------------------------------------------------------------------
summary_rows = [
    ("n_years",                              n),
    ("slope_a_deg_per_year",                 a),
    ("intercept_b_deg",                      b),
    ("R_squared",                            r2),
    ("sigma_residual_deg",                   sigma_res),
    ("SE_slope",                             SE_a),
    ("t_slope",                              t_a),
    ("p_value_slope_two_sided",              p_a),
    ("CI95_slope_low",                       CI_a[0]),
    ("CI95_slope_high",                      CI_a[1]),
    ("projection_year",                      TARGET_Y),
    ("projection_point_estimate_deg",        y_new),
    ("CI95_mean_low_deg",                    CI_yhat[0]),
    ("CI95_mean_high_deg",                   CI_yhat[1]),
    ("PI95_obs_low_deg",                     PI_obs[0]),
    ("PI95_obs_high_deg",                    PI_obs[1]),
    ("P_exceed_29_5_deg",                    p_exceed(29.5, y_new, SE_pred)),
    ("P_exceed_30_0_deg",                    p_exceed(30.0, y_new, SE_pred)),
    ("P_exceed_30_5_deg",                    p_exceed(30.5, y_new, SE_pred)),
    ("P_exceed_31_0_deg",                    p_exceed(31.0, y_new, SE_pred)),
    ("MannKendall_S",                        int(S)),
    ("MannKendall_Z",                        Z),
    ("MannKendall_p_value_two_sided",        p_MK),
    ("TheilSen_slope_deg_per_year",          sen_slope),
]
summary_df = pd.DataFrame(summary_rows, columns=["metric", "value"])
csv_out = os.path.join(fig_dir, f"sst_july_uncertainty_summary_{TARGET_Y}_{ts}.csv")
summary_df.to_csv(csv_out, index=False)

# Also export the raw July annual series so reviewers can replicate exactly
series_out = os.path.join(fig_dir, f"sst_july_annual_means_for_regression_{ts}.csv")
annual_series.to_csv(series_out, index=False)

print(f"\n✓ Figure saved: {png}")
print(f"✓ Summary CSV : {csv_out}")
print(f"✓ Raw series  : {series_out}")
print("\n✅ Done — these are the numbers reported in Annex I §§ 1.4.1–1.4.3.")

# =========================================================================
# 9) GERADOR DE RELATÓRIO AUTOMÁTICO (Adicionar no final do Código 13)
# =========================================================================

report_out = os.path.join(fig_dir, f"sst_july_scientific_report_{TARGET_Y}_{ts}.txt")

markdown_content = f"""# RELATÓRIO CIENTÍFICO: QUANTIFICAÇÃO DE INCERTEZA TSM (JULHO)
Gerado em: {datetime.now().strftime('%d/%m/%Y às %H:%M:%S')}
Configuração: TARGET_YEAR = {TARGET_Y} | MONTH = {MONTH}

-------------------------------------------------------------------------
1. REGRESSÃO LINEAR (OLS)
-------------------------------------------------------------------------
- Tamanho da Amostra (n): {n} anos
- Taxa de Aquecimento (Slope a): {a:.6f} °C/ano
- Intercepto (b): {b:.4f} °C
- Coeficiente de Determinação (R²): {r2:.4f}
- Erro Padrão Residual (σ_res): {sigma_res:.4f} °C
- Erro Padrão da Inclinação (SE_a): {SE_a:.6f}
- Estatística t: {t_a:.3f} (df={n-2})
- Valor-p do t-test: {p_a:.4f}
- Intervalo de Confiança 95% da Inclinação: [{CI_a[0]:.5f}, {CI_a[1]:.5f}] °C/ano

-------------------------------------------------------------------------
2. PROJEÇÃO E INTERVALOS PARA JULHO DE {TARGET_Y}
-------------------------------------------------------------------------
- Estimativa Pontual (ŷ): {y_new:.3f} °C
- 95% Intervalo de Confiança (Média): [{CI_yhat[0]:.3f}, {CI_yhat[1]:.3f}] °C
- 95% Intervalo de Previsão (Evento Único): [{PI_obs[0]:.3f}, {PI_obs[1]:.3f}] °C
  *(Nota: Use o Intervalo de Previsão para discussões de gerenciamento de risco)*

-------------------------------------------------------------------------
3. PROBABILIDADES DE EXCEDÊNCIA DE LIMIARES (JULHO DE {TARGET_Y})
-------------------------------------------------------------------------
"""

# Adiciona os limiares dinamicamente
for th in thresholds:
    p = p_exceed(th, y_new, SE_pred)
    marker = ""
    if th == 30.0: marker = " (Event temperature)"
    if th == 31.0: marker = " (World Aquatics Operational Threshold)"
    markdown_content += f"- P(SST > {th:.1f} °C) = {p:.4f} ({p*100:.2f}%){marker}\n"

markdown_content += f"""
-------------------------------------------------------------------------
4. TESTES NÃO-PARAMÉTRICOS DE ROBUSTEZ
-------------------------------------------------------------------------
- Estatística S de Mann-Kendall: {int(S)}
- Escore Z de Mann-Kendall: {Z:.3f}
- Valor-p (Mann-Kendall): {p_MK:.4f}
- Inclinação de Theil-Sen: {sen_slope:.6f} °C/ano

-------------------------------------------------------------------------
Fim do Relatório — Dados em conformidade com o Anexo I, §§ 1.4.1–1.4.3.
"""

# Escrita física do arquivo no Google Drive
with open(report_out, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"✓ Relatório Científico em formato texto salvo com sucesso:")
print(f"  📂 Caminho: {report_out}")